In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.enviroment-config

In [0]:
%run ../00-common/04.gold_helpers

In [0]:
target_table=f"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
from pyspark.sql import functions as F

In [0]:
results_df=(spark.table(f"{catalog_name}.{silver_schema}.results")
            .filter(F.col("batch_id")==v_batch_id)
            .withColumn("session_type",F.lit("RACE"))
            .drop("race_name","race_date","ingestion_timestamp","source_file","batch_id","batch_id","created_timestamp","updated_timestamp")
            )




In [0]:
sprints_df=(spark.table(f"{catalog_name}.{silver_schema}.sprints")
            .filter(F.col("batch_id")==v_batch_id)
            .withColumn("session_type",F.lit("SPRINT"))
            .drop("race_name","race_date","ingestion_timestamp","source_file","batch_id","created_timestamp","updated_timestamp")
            )




In [0]:
#UNION by name RESULTS AND SPRINTS
results_sprints_df=results_df.unionByName(sprints_df)

In [0]:
fact_session_results_df=(
    results_sprints_df
    .withColumn("is_wim",F.col("final_position")==1)
    .withColumn("is_podium",F.col("final_position").between(1,3))
    .withColumn("has_points",F.col("points")>0)
    )


In [0]:
display(fact_session_results_df.filter("season==2022"))


In [0]:
# (
#     fact_session_results_df
#     .write
#     .format("delta")
#     .mode("overwrite")
#     .saveAsTable(target_table)
# )

In [0]:
write_to_gold(
    input_df=fact_session_results_df,
    target_table=target_table,
    merge_condition="t.season=s.season AND t.driver_id=s.driver_id AND t.round=s.round AND t.session_type=s.session_type AND t.constructor_id=s.constructor_id",
    columns_to_update=["grid_position","completed_laps","car_number","final_position","finaal_position_text","points","status","is_wim","is_podium","has_points"] 
)

In [0]:
display(spark.table(target_table))